In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset"):
    print("📁", root)
    for d in dirs:
        print("   -", d)

📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset
   - valid
   - test
   - train
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/valid
   - wildfire
   - nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/valid/wildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/valid/nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/test
   - wildfire
   - nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/test/wildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/test/nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/train
   - wildfire
   - nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/train/wildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/train/nowildfire


# 1. Configuration
## Import Libraries
This includes Tensorflow Keras items for deeplearning framework, CNN backbone (ResNet50), model construction, training control and image loading pipelines

In [2]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.data.experimental import ignore_errors
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications.resnet50 import preprocess_input

print("Libraries installed successfully!")

2026-03-19 22:51:56.168267: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773960716.346364      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773960716.398897      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773960716.836592      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773960716.836645      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773960716.836648      24 computation_placer.cc:177] computation placer alr

Libraries installed successfully!


## General Parameters
Parameters that control data loading, model building and training.

In [3]:
BASE_DIR = "/kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset"
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE


# 2. Data Loading

In [4]:
train_ds = image_dataset_from_directory(
                os.path.join(BASE_DIR, "train"),
                batch_size=BATCH_SIZE,
                image_size=IMAGE_SIZE,
                label_mode="binary",
                shuffle=True,
                seed=SEED
                )
val_ds = image_dataset_from_directory(
                os.path.join(BASE_DIR, "valid"),
                batch_size=BATCH_SIZE,
                image_size=IMAGE_SIZE,
                label_mode="binary",
                shuffle=False
                )
test_ds = image_dataset_from_directory(
                os.path.join(BASE_DIR, "test"),
                batch_size=BATCH_SIZE,
                image_size=IMAGE_SIZE,
                label_mode="binary",
                shuffle=False
                )

Found 30250 files belonging to 2 classes.


I0000 00:00:1773960747.510396      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 6300 files belonging to 2 classes.
Found 6300 files belonging to 2 classes.


# Exploratory Data Analysis (EDA)
To be included

# Data Augmentation

## Ignore Errors / Prefetch
This skips bad files instead of crashing. ***In a future edit, this will be replaced with EDA and proper preprocessing.***

In [5]:
train_ds = train_ds.apply(ignore_errors())
val_ds = val_ds.apply(ignore_errors())
test_ds = test_ds.apply(ignore_errors())

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


## Augmentation
These small, random transformations will result in a more robust training, allowing the model to adapt to many different ways a wildfire may look.

In [6]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(.05),
    tf.keras.layers.RandomZoom(.1),
    tf.keras.layers.RandomContrast(.1),
])

# Build the Model

***NOTE: Add learning rate schedule (eg. lr_callback = tf.keras.callbacks.ReduceLROnPlateau(--hp))***

In [7]:
# Create the Input Tensor
inputs = Input(shape=(224, 224, 3))

#Load ResNet50 Without Classifier Head
base_model = ResNet50(weights="imagenet", include_top=False, input_tensor=inputs)

#Build Custom Classifier Head
x = data_augmentation(inputs)
x = preprocess_input(x)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(.2)(x)
outputs = Dense(1, activation="sigmoid")(x)

#Fine-tune Top Layers
base_model.trainable = True

for layer in base_model.layers[:-30]: #This freezes the backbone layers to preserve general visual features
    layer.trainable = False

model=Model(inputs=inputs, outputs=outputs)
model.compile(                               #This trains the top layers with small learning rate
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


# Train the Model

In [8]:
callbacks = [
    EarlyStopping(patience=3, 
                  verbose=1, 
                  restore_best_weights=True
                 ),
    ModelCheckpoint("best_resnet50_model.keras",
                   save_best_only=True,
                   verbose=1),
    ReduceLROnPlateau(monitor="val_loss",
                     factor=.2,
                     patience=2,
                     verbose=1)    
]

model_fit = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10


I0000 00:00:1773960769.627085      71 service.cc:152] XLA service 0x7dc4b4002f20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773960769.627119      71 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773960772.281341      71 cuda_dnn.cc:529] Loaded cuDNN version 91002


      2/Unknown 21s 72ms/step - accuracy: 0.6094 - loss: 0.7187

I0000 00:00:1773960780.198483      71 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    342/Unknown 43s 65ms/step - accuracy: 0.8430 - loss: 0.3538

...

944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.9952 - loss: 0.0156
Epoch 4: val_loss did not improve from 0.05155
945/945 ━━━━━━━━━━━━━━━━━━━━ 71s 75ms/step - accuracy: 0.9952 - loss: 0.0156 - val_accuracy: 0.9832 - val_loss: 0.0566 - learning_rate: 1.0000e-05
Epoch 5/10
352/945 ━━━━━━━━━━━━━━━━━━━━ 39s 66ms/step - accuracy: 0.9984 - loss: 0.0074

2026-03-19 22:58:30.594303: E tensorflow/core/lib/jpeg/jpeg_mem.cc:329] Premature end of JPEG data. Stopped at line 46/350


944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.9983 - loss: 0.0077
Epoch 5: val_loss did not improve from 0.05155

Epoch 5: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.
945/945 ━━━━━━━━━━━━━━━━━━━━ 71s 75ms/step - accuracy: 0.9983 - loss: 0.0077 - val_accuracy: 0.9841 - val_loss: 0.0608 - learning_rate: 1.0000e-05
Epoch 6/10
345/945 ━━━━━━━━━━━━━━━━━━━━ 39s 66ms/step - accuracy: 0.9989 - loss: 0.0056

2026-03-19 22:59:41.370432: E tensorflow/core/lib/jpeg/jpeg_mem.cc:329] Premature end of JPEG data. Stopped at line 46/350


944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.9990 - loss: 0.0052
Epoch 6: val_loss did not improve from 0.05155
945/945 ━━━━━━━━━━━━━━━━━━━━ 71s 75ms/step - accuracy: 0.9990 - loss: 0.0052 - val_accuracy: 0.9851 - val_loss: 0.0589 - learning_rate: 2.0000e-06
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


# Next Steps
- Evaluation
- Visualization / Transparency
- Summarization
- Annotation